Hacemos un append de la data de la ejecución de hoy (04/08/2026) en un volumen de databricks.
La data se lee desde el mismo volumen pero podría provenir de una fuente externa (Cómo S3 o Google Cloud Storage) y aplicaría una solución equivalente.

El rol de esta etapa es unificar la información extraida de distintas fuentes y mantener un registro historico de la información de todas las cargas la cuál este particionada por mes y año.

In [0]:
dataset_path = "/Volumes/iol_challenge/bronze/transaction_data/data_to_land/2026/08/04/challenge-iol-data-set.csv"
target_path = "iol_challenge.bronze.raw_ingestion"

df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(dataset_path)
    )

Enriquecemos la data de bronce con checkeos de calidad, columnas para particionamiento y el timestamp de ingesta

In [0]:
from pyspark.sql.functions import year, month, col, now, array_remove, array, when, expr, length, size, to_json, struct, lit

(
    df
        .withColumn("data_type", lit("transaction"))
        .withColumn("data", to_json(struct(*df.columns), options={"ignoreNullFields": "true"}))
        .withColumn("anio", year(col("fecha")))
        .withColumn("mes", month(col("fecha")))
        .withColumn("dia", month(col("fecha")))
        .withColumn("timestamp_ejecucion", now())
        .withColumn("errores_calidad", array_remove(
                array(
                    when(col("id_transaccion").isNull(), "ID_DE_TRANSACCION_NULO"),
                    when(length(col("id_transaccion")) != 14, "LONGITUD_DE_TRANSACCION_INVALIDA"),
                    when(col("fecha") > expr("current_timestamp()"), "FECHA_INVALIDA")
                ),
                None 
            )
        ).withColumn("tiene_errores_calidad",
            col("errores_calidad").isNotNull()
        ).select(
            col("data_type"),
            col("data"),
            col("dia"),
            col("anio"),
            col("mes"),
            col("timestamp_ejecucion"),
            col("errores_calidad"),
            col("tiene_errores_calidad"),
        )
        .write
        .format("delta")      
        .mode("append")
        .partitionBy("anio", "mes", "dia")
        .saveAsTable(target_path)
)

Verificación del volumen con información de la capa bronce ya generado

In [0]:
%sql
SELECT * FROM iol_challenge.bronze.raw_ingestion LIMIT 10;

In [0]:
%sql
WITH freq AS (
    SELECT get_json_object(data, '$.id_transaccion'), COUNT(*) as cantidad FROM iol_challenge.bronze.raw_ingestion WHERE data_type = 'transaction'  GROUP BY get_json_object(data, '$.id_transaccion')
) SELECT COUNT(*), cantidad FROM freq GROUP BY cantidad;

In [0]:
%sql
SELECT COUNT(*), tiene_errores_calidad FROM iol_challenge.bronze.raw_ingestion GROUP BY tiene_errores_calidad;